In [0]:
%run /Workspace/Users/erica.roque@gmail.com/Databricks-Certified-Data-Engineer-Associate/Includes/Copy-Datasets

In [0]:
%sql
use catalog demo_workspace;
use schema book_store


In [0]:
file = dbutils.fs.ls(f"{dataset_bookstore}/orders-raw")
display(file)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col

(spark.readStream
        .format("cloudFiles") #mecanismo de leitura
        .option("cloudFiles.format", "parquet") #formato do arquivo
        .option("cloudFiles.schemaLocation", "/Volumes/demo_workspace/book_store/books/checkpoints/orders_raw")
        .load(f"{dataset_bookstore}/orders-raw")
        .withColumn("ingestion_date", current_timestamp())
        .withColumn("source_file", col("_metadata.file_name"))
        .createOrReplaceTempView("orders_raw_temp")
)

In [0]:
df= spark.table("orders_raw_temp")
display(df, checkpointLocation="/Volumes/demo_workspace/book_store/books/checkpoints/orders_raw_2")

In [0]:
(spark.table("orders_raw_temp")
      .writeStream
      .format("delta")
      .option("checkpointLocation", "/Volumes/demo_workspace/book_store/books/checkpoints/orders_bronze_")
      .outputMode("append")
      .table("orders_bronze_new")
 
 )

In [0]:
%sql
select * from orders_bronze_new


In [0]:
load_new_data()

In [0]:
(spark.read
       .format("json")
        .load(f"{dataset_bookstore}/customers-json")
        .createOrReplaceTempView("customers_lookup")
)


In [0]:
%sql
select * from customers_lookup

In [0]:
(spark.readStream
      .table("orders_bronze_new")  
      .createOrReplaceTempView("orders_bronze_tmp")
)


In [0]:
from pyspark.sql.functions import col, cast, explode, get_json_object


df_orders = spark.table("orders_bronze_tmp")
df_custumers = spark.table("customers_lookup")
df_orders_enriched = (
    df_orders.alias("o")
    .join(
        df_custumers.alias("c"), 
        col("o.customer_id") == col("c.customer_id"), 
        "inner"
    )
    .filter(col("o.quantity")> 0)
    .select(
        col("c.email").alias("customer_email"),
        col("c.customer_id").alias("customer_id"),
        col("o.quantity").alias("quantity"),
        get_json_object(col("c.profile"), "$.first_name").alias("first_name"),
        get_json_object(col("c.profile"), "$.last_name").alias("last_name"),
        col("o.order_timestamp").cast("timestamp").alias("order_timestamp"),
        col("o.books").alias("books")
    )
)
df_orders_enriched.createOrReplaceTempView("orders_enriched_tmp")




In [0]:
(spark.table("orders_enriched_tmp")
      .writeStream
      .format("delta")
      .option("checkpointLocation", "/Volumes/demo_workspace/book_store/books/checkpoints/orders_silver")
      .outputMode("append")
      .table("order_silver")  
 )

In [0]:
%sql
select count(*) from order_silver

In [0]:
load_new_data()

In [0]:
(spark.readStream
    .table("order_silver")
    .createOrReplaceTempView("orders_silver_tmp")
)


In [0]:
df= spark.table("orders_silver_tmp")
display(df, checkpointLocation="/Volumes/demo_workspace/book_store/books/checkpoints/orders_silver_layer_new")



In [0]:
from pyspark.sql.functions import col, date_trunc, sum

df = (
    spark.table("orders_silver_tmp")
    .select(
        col("customer_id"),
        col("first_name"),
        col("last_name"),
        col("order_timestamp"),
        col("quantity")
    )
    .withColumn("order_date", date_trunc("DD", col("order_timestamp")))
    .groupBy("customer_id", "first_name", "last_name", "order_date")
    .agg(
        sum(col("quantity")).alias("book_count")
    )
)

df.createOrReplaceTempView("daily_customer_books")


In [0]:
df= spark.table("daily_customer_books")

display(df, checkpointLocation="/Volumes/demo_workspace/book_store/books/checkpoints/daily_customer_books_new")


In [0]:
(spark.table("daily_customer_books")
    .writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", "/Volumes/demo_workspace/book_store/books/checkpoints/daily_customer_books__")
   # .trigger(availableNow = True)
    .table("daily_customer_books_new")

)

In [0]:
%sql
select * from daily_customer_books_new

In [0]:
for s in spark.streams.active:
    print("Stoping stream:" + s.id)
    s.stop()
    s.awaitTermination() 